In [4]:
import argparse
import json
import os
import signal
import subprocess
import time
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
api_key = os.environ.get("GROQ_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

In [13]:
"""
capture_orchestrator.py
-----------------------
Orchestrates per-prompt network captures using tshark, then calls
each configured LLM API. Saves a .pcap file per session alongside
a metadata JSON for downstream feature extraction.
 
Requirements:
    pip install groq openai anthropic httpx requests
    sudo apt install tshark  (or brew install wireshark on macOS)
 
Usage:
    python capture_orchestrator.py --runs 3 --delay 2
"""
import argparse
import json
import os
import signal
import subprocess
import time
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional
from dotenv import load_dotenv
load_dotenv()

api_key = os.environ.get("GROQ_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# ── Optional SDK imports (only imported if key is set) ────────────────────────
try:
    from groq import Groq
except ImportError:
    Groq = None  # type: ignore
    
# try:
#     from google import genai
# except ImportError:
#     genai = None  # type: ignore
 
# try:
#     import openai
# except ImportError:
#     openai = None  # type: ignore

 
from prompt_bank import PROMPT_BANK, Prompt
 
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
 
OUTPUT_DIR = Path("captures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
# Network interface for tshark. Use "any" on Linux, or find yours with:
#   tshark -D
CAPTURE_INTERFACE = r"\Device\NPF_{2C768E8E-C50A-4740-ABB3-2AF5ADB15BA2}"
 
# Seconds to wait between sessions to let TCP flows settle
INTER_SESSION_DELAY = 2.0
 
# Map of LLM providers. Each entry needs a callable that takes a prompt string
# and returns (response_text, latency_seconds).
# Add/remove providers here; the rest of the pipeline is automatic.
LLM_CONFIGS: Dict[str, Dict[str, Any]] = {
    "groq_qwen3_32b": {
        "api_key_env": "GROQ_API_KEY",
        "model": "qwen/qwen3-32b",
        "provider": "groq",
    },
    # "groq_llama3_70b": {
    #     "api_key_env": "GROQ_API_KEY",
    #     "model": "llama3-70b-8192",
    #     "provider": "groq",
    # },
    # "openai_gpt4o": {
    #     "api_key_env": "OPENAI_API_KEY",
    #     "model": "gpt-4o",
    #     "provider": "openai",
    # },
}

 
# ─────────────────────────────────────────────────────────────────────────────
# DATA MODEL
# ─────────────────────────────────────────────────────────────────────────────
 
@dataclass
class SessionMeta:
    session_id: str
    llm_name: str
    model: str
    workload: str
    prompt_text: str
    expected_tokens: str
    tags: list
    timestamp_utc: str
    pcap_path: str
    response_text: str
    response_tokens_approx: int
    latency_seconds: float
    error: Optional[str]
 
 
# ─────────────────────────────────────────────────────────────────────────────
# LLM CALLERS
# ─────────────────────────────────────────────────────────────────────────────
 
def call_groq_stream(model: str, api_key: str, prompt: str):
    if Groq is None:
        raise ImportError("groq package not installed")

    client = Groq(api_key=api_key)

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    return stream
    
    #Unhashtag this section for full text responses 
#     response = client.chat.completions.create(
#         model=model,
#         messages=[{"role": "user", "content": prompt}],)
 
 
# def call_openai(model: str, api_key: str, prompt: str):
#     if openai is None:
#         raise ImportError("openai package not installed")
#     client = openai.OpenAI(api_key=api_key)
#     t0 = time.perf_counter()
#     response = client.chat.completions.create(
#         model=model,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     latency = time.perf_counter() - t0
#     text = response.choices[0].message.content or ""
#     return text, latency

 
CALLER_MAP = {
    "groq": call_groq_stream,
    # "openai": call_openai,
    # "anthropic": call_anthropic,
}
 
 
# ─────────────────────────────────────────────────────────────────────────────
# TSHARK HELPERS
# ─────────────────────────────────────────────────────────────────────────────
 
def start_capture(pcap_path: Path) -> subprocess.Popen:
    cmd = [
        r"C:\Program Files\Wireshark\tshark.exe",
        "-i", CAPTURE_INTERFACE,
        "-w", str(pcap_path),
        "-q",
    ]

    print("tshark command:", " ".join(cmd))

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    return proc
 
 
def stop_capture(proc: subprocess.Popen) -> None:
    """Gracefully stop tshark and wait for it to flush the pcap."""
    proc.send_signal(signal.SIGTERM)
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SESSION RUNNER
# ─────────────────────────────────────────────────────────────────────────────
 
def run_session(llm_name: str, cfg: Dict[str, Any], prompt: Prompt) -> SessionMeta:
    session_id = str(uuid.uuid4())[:8]
    ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    safe_llm = llm_name.replace("/", "_")
    
    pcap_filename = f"{ts}_{safe_llm}_{prompt.workload}_{session_id}.pcap"
    pcap_path = (OUTPUT_DIR / pcap_filename).resolve()
    print("PCAP path:", pcap_path)
 
    api_key = os.environ.get(cfg["api_key_env"], "")
    if not api_key:
        print(f"  [SKIP] {llm_name}: env var {cfg['api_key_env']} not set")
        return SessionMeta(
            session_id=session_id,
            llm_name=llm_name,
            model=cfg["model"],
            workload=prompt.workload,
            prompt_text=prompt.text,
            expected_tokens=prompt.expected_tokens,
            tags=prompt.tags,
            timestamp_utc=ts,
            pcap_path=str(pcap_path),
            response_text="",
            response_tokens_approx=0,
            latency_seconds=0.0,
            error=f"API key not set: {cfg['api_key_env']}",
        )
 
    caller = CALLER_MAP.get(cfg["provider"])
    if caller is None:
        raise ValueError(f"Unknown provider: {cfg['provider']}")
 
    print(f"  [{llm_name}] Capturing -> {pcap_filename}")
    proc = start_capture(pcap_path)
    time.sleep(2) 
 
    response_text = ""
    latency = 0.0
    error = None
    
    try:
        stream = caller(cfg["model"], api_key, prompt.text)
        token_series = []
        full_text = ""

        t0 = time.perf_counter()

        for chunk in stream:
            if chunk.choices[0].delta.content:
                now = time.perf_counter()
                token = chunk.choices[0].delta.content

                full_text += token
                token_series.append({"t_rel": now - t0,"token": token})
                #optinal:
                #print(token, end="", flush=True)

        latency = time.perf_counter() - t0
        response_text = full_text
        
        token_path = pcap_path.with_suffix(".tokens.json")
        with open(token_path, "w") as f:
            json.dump(token_series, f, indent=2)

    except Exception as exc:
        error = str(exc)
        print(f"API error: {exc}")
        
    finally:
        time.sleep(2) 
        stop_capture(proc)
 
    meta = SessionMeta(
        session_id=session_id,
        llm_name=llm_name,
        model=cfg["model"],
        workload=prompt.workload,
        prompt_text=prompt.text,
        expected_tokens=prompt.expected_tokens,
        tags=prompt.tags,
        timestamp_utc=ts,
        pcap_path=str(pcap_path),
        response_text=response_text,
        response_tokens_approx=len(response_text.split()),
        latency_seconds=round(latency, 4),
        error=error,
    )
 
    # Save metadata alongside the pcap
    
        
    meta_path = pcap_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(asdict(meta), f, indent=2)
 
    print(f"Latency={latency:.2f}s  ~{meta.response_tokens_approx} words")
    return meta
 
 
# ─────────────────────────────────────────────────────────────────────────────
# MAIN ORCHESTRATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
 
def run_all(runs: int = 1, delay: float = INTER_SESSION_DELAY,
            llm_filter: Optional[str] = None,
            workload_filter: Optional[str] = None) -> None:
    """
    Iterate over every (LLM x prompt) pair, repeated `runs` times.
    Optionally filter by llm_name or workload type.
    """
    targets = {k: v for k, v in LLM_CONFIGS.items()
               if llm_filter is None or llm_filter in k}
    prompts = [p for p in PROMPT_BANK
               if workload_filter is None or p.workload == workload_filter]
 
    total = len(targets) * len(prompts) * runs
    print(f"\nStarting capture run: {len(targets)} LLMs x {len(prompts)} prompts x {runs} run(s) = {total} sessions\n")
 
    completed = 0
    for run_idx in range(runs):
        print(f"── Run {run_idx + 1}/{runs} ─────────────────────────────────────────")
        for llm_name, cfg in targets.items():
            for prompt in prompts:
                run_session(llm_name, cfg, prompt)
                completed += 1
                print(f"  Progress: {completed}/{total}")
                time.sleep(delay)
 
    print(f"\nDone. Captures saved to: {OUTPUT_DIR.resolve()}")
 
 
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="LLM traffic capture orchestrator")
    parser.add_argument("--runs", type=int, default=1,
                        help="Number of repetitions per (LLM, prompt) pair")
    parser.add_argument("--delay", type=float, default=INTER_SESSION_DELAY,
                        help="Seconds between sessions")
    parser.add_argument("--llm", type=str, default=None,
                        help="Filter to LLM names containing this string")
    parser.add_argument("--workload", type=str, default=None,
                        help="Filter to a specific workload type")
    import sys

    if "ipykernel" in sys.modules:
        args = parser.parse_args(args=[])
    else:
        args = parser.parse_args()
 
    run_all(runs=args.runs, delay=args.delay, llm_filter=args.llm, workload_filter=args.workload)


Starting capture run: 1 LLMs x 42 prompts x 1 run(s) = 42 sessions

── Run 1/1 ─────────────────────────────────────────
PCAP path: E:\AS9Wa\MSc Thesis\AgenticVision\src\captures\20260415T210924Z_groq_qwen3_32b_short_factual_87288e8d.pcap
  [groq_qwen3_32b] Capturing -> 20260415T210924Z_groq_qwen3_32b_short_factual_87288e8d.pcap
tshark command: C:\Program Files\Wireshark\tshark.exe -i \Device\NPF_{2C768E8E-C50A-4740-ABB3-2AF5ADB15BA2} -w E:\AS9Wa\MSc Thesis\AgenticVision\src\captures\20260415T210924Z_groq_qwen3_32b_short_factual_87288e8d.pcap -q
Latency=0.52s  ~192 words
  Progress: 1/42
PCAP path: E:\AS9Wa\MSc Thesis\AgenticVision\src\captures\20260415T210931Z_groq_qwen3_32b_short_factual_ac4c32fa.pcap
  [groq_qwen3_32b] Capturing -> 20260415T210931Z_groq_qwen3_32b_short_factual_ac4c32fa.pcap
tshark command: C:\Program Files\Wireshark\tshark.exe -i \Device\NPF_{2C768E8E-C50A-4740-ABB3-2AF5ADB15BA2} -w E:\AS9Wa\MSc Thesis\AgenticVision\src\captures\20260415T210931Z_groq_qwen3_32b_shor